In [ ]:

# -- Cell 1 -- rclone + Drive. Same pattern as every other notebook here.
# Requires: Settings -> Internet ON, Accelerator GPU T4 x2, RCLONE_DRIVE_TOKEN
# attached to THIS notebook.
import os, subprocess

r = subprocess.run("curl -s https://rclone.org/install.sh | sudo bash", shell=True)
if r.returncode not in (0, 3):
    raise RuntimeError("rclone install failed (exit %d)" % r.returncode)

from kaggle_secrets import UserSecretsClient
token = UserSecretsClient().get_secret("RCLONE_DRIVE_TOKEN")
os.makedirs("/root/.config/rclone", exist_ok=True)
with open("/root/.config/rclone/rclone.conf", "w") as f:
    f.write("[drive]\ntype = drive\nscope = drive\ntoken = " + token + "\n")

REMOTE = "drive:Distillation"
out = subprocess.run("rclone lsf " + REMOTE, shell=True, capture_output=True, text=True)
print(out.stdout or out.stderr)
assert out.returncode == 0, "cannot see " + REMOTE


In [ ]:

# -- Cell 2 -- deps + GPUs.
#
# THEIR classification script is already a LoRA script. r=16, alpha=32,
# dropout 0.1, target_modules=["qkv_proj"], head Linear(d,d)->SiLU->Dropout->
# Linear(d,1), BCEWithLogitsLoss, AdamW lr 3e-4, linear warmup 10%. Nothing here
# reimplements any of that -- the only thing this notebook supplies is which
# backbone weights to load, which is exactly the variable under test.
subprocess.run('pip install -q -U "transformers>=5.0" peft lightning', shell=True, check=True)
subprocess.run("pip uninstall -y -q torchao", shell=True)   # peft/torchao clash guard

import torch, numpy as np, pandas as pd, glob, json, time
print("torch", torch.__version__, "| GPUs", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print("   cuda:%d %s %.0f GB" % (i, p.name, p.total_memory / 1e9))
NGPU = max(1, torch.cuda.device_count())
if NGPU < 2:
    print("\nonly one GPU -- jobs run sequentially, roughly double the wall clock")


In [ ]:

# -- Cell 3 -- their code and data.
#
# Only their_repo/data and their_repo/training come down. their_repo is 468 files
# / 490 MB and 401 of those (452 MB) are figure_generation/, which nothing here
# reads. Drive charges per file, so this is seconds instead of a long wait.
PROJ = "/kaggle/working/project"
REPO = PROJ + "/their_repo"
os.makedirs(REPO, exist_ok=True)

for sub in ("data", "training"):
    if os.path.isdir(REPO + "/" + sub):
        print(sub, ": already present"); continue
    subprocess.run("rclone copy %s/their_repo/%s %s/%s --transfers 16 --checkers 16 -P"
                   % (REMOTE, sub, REPO, sub), shell=True, check=True)

TRAIN_PY = REPO + "/training/02_classification_benchmarks_training_code/scripts/classification_finetuning_v2.py"
DATA_DIR = REPO + "/data"
for p in (TRAIN_PY, DATA_DIR + "/CellPPD_train.csv", DATA_DIR + "/CellPPD_test.csv"):
    assert os.path.exists(p), "missing: " + p
    print("ok", p.replace(PROJ + "/", ""))

tr = pd.read_csv(DATA_DIR + "/CellPPD_train.csv")
te = pd.read_csv(DATA_DIR + "/CellPPD_test.csv")
print("\ntrain %s  %s" % (tr.shape, dict(tr.label.value_counts())))
print("test  %s  %s" % (te.shape, dict(te.label.value_counts())))
print("\nNo CellPPD_val.csv exists, so their script takes its 5-fold CV branch:")
print("each fold trains on 4/5 of train and predicts the FULL test set, and the")
print("five prediction sets are ensembled in Cell 7. Same path that produced the")
print("published numbers.")


In [ ]:

# -- Cell 4 -- our code, the released 32M init, and the two trained checkpoints.
#
# Checkpoints are the arms under test:
#   treatment  cached KD + MTR + SPKD, top-16 targets, one fixed mask per molecule
#   kdlive     pure Hinton KD, live teacher, full 405-way targets, fresh mask/epoch
#
# WARM-START IS INCLUDED AS A THIRD ARM even though it was not asked for, because
# without it the other two are uninterpretable: "treatment 0.87 vs kdlive 0.86"
# cannot distinguish "distillation helped" from "both drifted down from where
# they started". It is their released 32M, untouched, so it costs one more job
# and turns a comparison into a measurement. Drop it from ARMS if you disagree.
CODE = "/kaggle/working/distill"
INIT = "/kaggle/working/models/peptideclm-2-mlm-small"
CKPT = "/kaggle/working/ckpt"
EXPORT = "/kaggle/working/exported"

def rlsf(path):
    r = subprocess.run("rclone lsf " + path, shell=True, capture_output=True, text=True)
    return r.stdout.split() if r.returncode == 0 else []

if not os.path.exists(CODE + "/export_student.py"):
    subprocess.run("rclone copy %s/distill %s --transfers 8 -P" % (REMOTE, CODE),
                   shell=True, check=True)

# The released 32M sits on Drive in flat layout; the 337M in HF cache layout.
# Try flat first, then snapshots, and say which path was missing rather than
# raising IndexError on an empty listing.
if not os.path.exists(INIT + "/model.safetensors"):
    flat = REMOTE + "/models/peptideclm-2-mlm-small"
    snaps = REMOTE + "/models/models--aaronfeller--peptideclm-2-mlm-small/snapshots"
    os.makedirs(INIT, exist_ok=True)
    if any(f.startswith("model.safetensors") for f in rlsf(flat)):
        subprocess.run("rclone copy %s %s -P" % (flat, INIT), shell=True, check=True)
    else:
        shas = [x.rstrip("/") for x in rlsf(snaps)]
        assert shas, "32M init not found. Tried  %s  and  %s" % (flat, snaps)
        subprocess.run("rclone copy %s/%s %s -P" % (snaps, shas[0], INIT),
                       shell=True, check=True)

# Trained checkpoints. student_final.pt, not latest.pt: latest.pt also carries
# optimizer state (3x the size) and both hold the same weights at the end.
PULL = {"treatment": "results/distill/treatment/student_final.pt",
        "kdlive": "results/kd_live/student_final.pt"}
os.makedirs(CKPT, exist_ok=True)
for name, remote in PULL.items():
    dest = "%s/%s.pt" % (CKPT, name)
    if not os.path.exists(dest):
        d, f = os.path.split(remote)
        subprocess.run("rclone copy %s/%s %s --include %s -P" % (REMOTE, d, CKPT, f),
                       shell=True, check=True)
        os.rename(os.path.join(CKPT, f), dest)
    print("%-10s %.2f GB" % (name, os.path.getsize(dest) / 1e9))


In [ ]:

# -- Cell 5 -- export each arm as a plain HuggingFace directory.
#
# Their script calls AutoModel.from_pretrained(model_name, trust_remote_code=True),
# so each arm has to look like a released checkpoint: backbone weights plus
# config.json / config.py / ChemPepMTR.py / tokenizer files in one flat folder.
# export_student.py handles both checkpoint layouts -- the cached run saved a
# Student wrapper (backbone.* + mtr_head.*, head dropped here), the live run saved
# a bare AutoModel -- and round-trips each export through from_pretrained before
# returning.
#
# THE FOLDER NAME IS LOAD-BEARING. Their script derives the output filename from
# model_name.split("/")[-1], so peptideclm-2-mlm-small-kdlive produces
# CellPPD_peptideclm-2-mlm-small-kdlive_results.csv. Two arms sharing a basename
# would overwrite each other's results with no error.
r = subprocess.run(["python", "export_student.py",
                    "--runs", "/nonexistent", "--arms", "",
                    "--init", INIT, "--out", EXPORT,
                    "--also", "treatment=%s/treatment.pt" % CKPT,
                    "--also", "kdlive=%s/kdlive.pt" % CKPT],
                   cwd=CODE, capture_output=True, text=True)
print(r.stdout[-2500:])
if r.returncode != 0:
    print("----- STDERR -----"); print(r.stderr[-2500:])
assert r.returncode == 0, "export failed"

ARMS = ["peptideclm-2-mlm-small-warmstart",
        "peptideclm-2-mlm-small-treatment",
        "peptideclm-2-mlm-small-kdlive"]
for m in ARMS:
    assert os.path.exists(EXPORT + "/" + m + "/model.safetensors"), "missing export " + m

# The arms must actually differ. An export bug that silently wrote the init three
# times would produce three identical MCCs and look like a null result.
from safetensors.torch import load_file
ref = load_file(EXPORT + "/" + ARMS[0] + "/model.safetensors")
print()
for m in ARMS[1:]:
    d = load_file(EXPORT + "/" + m + "/model.safetensors")
    diff = sum(1 for k in ref if not torch.equal(ref[k], d[k]))
    print("%-40s %3d/%d tensors differ from warm-start" % (m, diff, len(ref)))
    assert diff == len(ref), "%s is identical to the warm start -- export bug" % m


In [ ]:

# -- Cell 6 -- run their script, unmodified, once per (arm, seed).
#
# 3 arms x 3 seeds = 9 jobs, two at a time on the two T4s. Seeds 101/202/303 are
# the ones their own released runs used, so the spread here is comparable to the
# spread in their artifacts -- which matters, because the CellPPD reproduction
# established a noise floor of about +/-0.02 MCC. A single seed cannot separate
# arms that differ by less than that.
#
# batch_size 32 matches their run_classification_finetuning.sh. Everything else
# is their default: max_epochs 10, patience 5 on val_loss, lr 3e-4.
#
# --gpu_index MUST be passed. Their Trainer does devices=[int(args.gpu_index)]
# using the raw argument rather than the resolved `gpu`, and its default is None,
# so omitting it is a TypeError inside their code.
#
# Budget: ~930 train rows at batch 32 is 29 steps/epoch, 10 epochs, 5 folds, on a
# 32M model -- a few minutes per job, well under an hour for all nine.
#
# Lightning writes a checkpoint per fold under log_dir. Nine jobs x five folds at
# ~130 MB is 5.8 GB, so log_dir points at /tmp (not the 20 GB /kaggle/working
# quota) and each job's logs are deleted once its results CSV is safely written.
SEEDS = [101, 202, 303]
OUT = "/kaggle/working/results/cellppd_lora"
os.makedirs(OUT, exist_ok=True)

jobs = [(m, s) for m in ARMS for s in SEEDS]
todo = [(m, s) for m, s in jobs
        if not os.path.exists("%s/%s/seed_%d/CellPPD_%s_results.csv" % (OUT, m, s, m))]
print("%d jobs, %d still to run" % (len(jobs), len(todo)))

running, t0 = [], time.time()
free = list(range(NGPU))
while todo or running:
    while todo and free:
        m, s = todo.pop(0)
        gpu = free.pop(0)
        d = "%s/%s/seed_%d" % (OUT, m, s)
        os.makedirs(d, exist_ok=True)
        log = open("%s/train.log" % d, "w")
        cmd = ["python", TRAIN_PY, "--dataset", "CellPPD",
               "--gpu", "0", "--gpu_index", "0",     # index into CUDA_VISIBLE_DEVICES
               "--model_name", EXPORT + "/" + m,
               "--batch_size", "32", "--seed", str(s),
               "--data_dir", DATA_DIR, "--save_path", d,
               "--log_dir", "/tmp/logs/%s_%d" % (m, s)]
        p = subprocess.Popen(cmd, cwd=os.path.dirname(TRAIN_PY),
                             stdout=log, stderr=subprocess.STDOUT,
                             env=dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu)))
        running.append((m, s, gpu, p, d))
        print("[%5.1f min] launch %-40s seed %d on GPU %d" % ((time.time()-t0)/60, m, s, gpu))
    time.sleep(20)
    for job in list(running):
        m, s, gpu, p, d = job
        if p.poll() is None:
            continue
        running.remove(job); free.append(gpu)
        ok = p.returncode == 0 and glob.glob(d + "/*_results.csv")
        print("[%5.1f min] %-40s seed %d -> %s"
              % ((time.time()-t0)/60, m, s, "ok" if ok else "FAILED rc=%s" % p.returncode))
        if ok:
            subprocess.run("rm -rf /tmp/logs/%s_%d" % (m, s), shell=True)
        else:
            print("".join(open(d + "/train.log").readlines()[-25:]))
print("\nall jobs done in %.1f min" % ((time.time() - t0) / 60))


In [ ]:

# -- Cell 7 -- score.
#
# Their script writes raw logits only; the code that turned those into the paper's
# MCCs was never released. This is the aggregation reverse-engineered during the
# CellPPD reproduction from the artifacts they DID ship
# (figure_generation/results/runs_LoRA_highrank/cellppd_all.csv): each of the 5
# folds predicts the full test set, ensemble by MEAN LOGIT, threshold at 0.
#
# CALIBRATION: no aggregation reproduces their shipped file exactly. Mean-logit
# lands 0.009 off on average, 0.020 worst case. So differences between arms below
# ~0.02 MCC are inside the noise and should not be read as real -- that is the
# whole reason for three seeds.
from sklearn.metrics import matthews_corrcoef, roc_auc_score, accuracy_score, f1_score

rows = []
for f in sorted(glob.glob(OUT + "/*/seed_*/CellPPD_*_results.csv")):
    seed = int(os.path.basename(os.path.dirname(f)).split("_")[1])
    arm = os.path.basename(os.path.dirname(os.path.dirname(f)))
    d = pd.read_csv(f)
    d["i"] = d.groupby("fold").cumcount()          # align the 5 copies of the test set
    g = d.groupby("i")
    y = g.true_label.first().values
    p = g.predicted_label.mean().values            # ensemble = mean logit
    rows.append(dict(arm=arm.replace("peptideclm-2-mlm-small-", ""), seed=seed,
                     mcc=matthews_corrcoef(y, (p > 0).astype(int)),
                     auc=roc_auc_score(y, p),
                     acc=accuracy_score(y, (p > 0).astype(int)),
                     f1=f1_score(y, (p > 0).astype(int))))
res = pd.DataFrame(rows).sort_values(["arm", "seed"])
print(res.to_string(index=False))

agg = res.groupby("arm").mcc.agg(["mean", "std", "count"])
print("\n=== test MCC over %d seeds ===" % len(SEEDS))
print(agg.to_string())

if "warmstart" in agg.index:
    base = agg.loc["warmstart", "mean"]
    print("\ndistillation effect (arm - warm-start), noise floor ~0.02:")
    for a in agg.index:
        if a != "warmstart":
            print("   %-10s %+.4f" % (a, agg.loc[a, "mean"] - base))

res.to_csv(OUT + "/cellppd_lora_metrics.csv", index=False)
DEST = REMOTE + "/results/cellppd_lora"
subprocess.run("rclone copy %s %s --drive-chunk-size 64M -P" % (OUT, DEST),
               shell=True, check=True)
print("\nuploaded to " + DEST)
